In [1]:
import sys
import os
from pathlib import Path
# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)


import time
import re
import requests
import json
import chromadb
from typing import List, Dict, Any


from configs.setting import settings
from configs.GetConfig import config
from src.a_ingestion.a1_loader import SupabaseDataLoader
from src.b_indexing.b0_vector_db import ChromaVectorDatabase
from src.b_indexing.b1_embedding import EmbeddingService
from src.b_indexing.b2_rerank import RerankService


In [2]:
SuperbaseDataLoader = SupabaseDataLoader()
products = SuperbaseDataLoader.load_products()
policies = SuperbaseDataLoader.load_policies()

[Loader] Đang tải dữ liệu sản phẩm từ Supabase...
  -> Đã tải batch 0 - 999 (1000 sản phẩm)
  -> Đã tải batch 1000 - 1324 (325 sản phẩm)
[Loader] Đã tải thành công tổng cộng 1325 sản phẩm.
[Loader] Đang tải dữ liệu chính sách từ Supabase...
[Loader] Đã tải thành công 3 chính sách từ database.


In [3]:
# Khởi tạo kết nối ChromaDB
client = ChromaVectorDatabase()

# Tách biệt làm 2 Collection chuyên biệt
product_col = client.get_or_create_collection(name="products_collection")
policy_col = client.get_or_create_collection(name="policies_collection")

print(f"Tổng vector sản phẩm: {product_col.count()}")
print(f"Tổng vector chính sách: {policy_col.count()}")

Tổng vector sản phẩm: 1325
Tổng vector chính sách: 3


In [9]:
from src.b_indexing.b0_vector_db import ChromaVectorDatabase

# 1. Kết nối ChromaDB
db = ChromaVectorDatabase()

# 2. Lấy đối tượng collection thô
product_col = db.get_collection("products_collection")
policy_col = db.get_collection("policies_collection")

# ==========================================
# 📊 KIỂM TRA PRODUCTS_COLLECTION (SẢN PHẨM)
# ==========================================
print("=== Cấu trúc trường trong PRODUCTS_COLLECTION ===")
# Lấy 1 bản ghi đầu tiên làm mẫu
prod_sample = product_col.get(limit=1)

if prod_sample["ids"]:
    print(f"1. Các cột dữ liệu cơ bản của ChromaDB: {list(prod_sample.keys())}")
    # Cột cơ bản thường gồm: ['ids', 'embeddings', 'metadatas', 'documents', 'data', 'uris']
    
    print(f"2. Các trường thuộc tính (Metadata) của Sản phẩm: {list(prod_sample['metadatas'][0].keys())}")
    # Thường gồm các cột từ Supabase như: ['name', 'price', 'category_id', 'description', ...]
    
    print("\nVí dụ 1 bản ghi sản phẩm mẫu:")
    print(f"- ID: {prod_sample['ids'][0]}")
    print(f"- Metadata: {prod_sample['metadatas'][0]}")
    print(f"- Document (Text nhúng): {prod_sample['documents'][0][:200]}...")
else:
    print("Collection sản phẩm đang trống!")

print("\n" + "="*60 + "\n")

# ==========================================
# 📊 KIỂM TRA POLICIES_COLLECTION (CHÍNH SÁCH)
# ==========================================
print("=== Cấu trúc trường trong POLICIES_COLLECTION ===")
policy_sample = policy_col.get(limit=1)

if policy_sample["ids"]:
    print(f"1. Các cột dữ liệu cơ bản của ChromaDB: {list(policy_sample.keys())}")
    print(f"2. Các trường thuộc tính (Metadata) của Chính sách: {list(policy_sample['metadatas'][0].keys())}")
    
    print("\nVí dụ 1 bản ghi chính sách mẫu:")
    print(f"- ID: {policy_sample['ids'][0]}")
    print(f"- Metadata: {policy_sample['metadatas'][0]}")
    print(f"- Document (Text nhúng): {policy_sample['documents'][0][:200]}...")
else:
    print("Collection chính sách đang trống!")


=== Cấu trúc trường trong PRODUCTS_COLLECTION ===
1. Các cột dữ liệu cơ bản của ChromaDB: ['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas']
2. Các trường thuộc tính (Metadata) của Sản phẩm: ['brand', 'price', 'category', 'product_id']

Ví dụ 1 bản ghi sản phẩm mẫu:
- ID: prod_136045
- Metadata: {'brand': 'Nubia', 'price': 13990000.0, 'category': 'phone', 'product_id': '136045'}
- Document (Text nhúng): Sản phẩm: Nubia Neo 5 GT Special Edition 12GB 256GB

Thương hiệu: Nubia | Danh mục: phone

Thông tin giá & Kho hàng:
- Giá thực tế: 13,990,000 VNĐ
- Giá gốc niêm yết: 13,990,000 VNĐ
- Mức giảm giá: Kh...


=== Cấu trúc trường trong POLICIES_COLLECTION ===
1. Các cột dữ liệu cơ bản của ChromaDB: ['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas']
2. Các trường thuộc tính (Metadata) của Chính sách: ['category', 'chunk_index', 'title', 'policy_id']

Ví dụ 1 bản ghi chính sách mẫu:
- ID: policy_0d255481-edd4-44ab-bb06-0f9c6b1a2de5_chunk_0
- M

# 1. Test basic retrival 

In [ ]:
embedding_service = EmbeddingService(config=config, settings=settings)
rerank_service = RerankService(config=config, settings=settings)

user_query = "Laptop HP dưới 20 triệu"

embedding_text = embedding_service.get_embedding(user_query, max_retries=3)

# Tìm kiếm thô trên ChromaDB 
raw_results = client.query(
    collection_name="products_collection", 
    query_embeddings=[embedding_text],
    n_results=config.retrieval.product.k_query, 
)

# =====================================================

print(f"=== [LẦN 1] KẾT QUẢ TRUY VẤN THÔ TỪ CHROMADB (K_QUERY = {config.retrieval.product.k_query}) ===")
for i in range(len(raw_results['ids'][0])):
    doc_id = raw_results['ids'][0][i]
    document_content = raw_results['documents'][0][i]
    distance = raw_results['distances'][0][i]
    metadata = raw_results['metadatas'][0][i]
    
    # Trích xuất tên sản phẩm từ dòng đầu tiên của document_content
    product_name = document_content.split('\n')[0].replace("Sản phẩm:", "").strip()
    
    print(f"Top {i+1} [ID: {doc_id}] - Khoảng cách cosine (Cosine Distance): {distance:.4f}")
    print(f"=> Tên sản phẩm: {product_name}")  # Đã sửa
    print(f"=> Giá bán: {metadata.get('price')} VND")
    print(f"=> Nội dung chunk:\n{document_content}")
    print("-" * 50)
    print("\n" + "="*80 + "\n")

# =====================================================

# Định dạng lại documents thô thành cấu trúc list các dict {"text": ...} để gửi cho API Rerank
rerank_inputs = [{"text": doc} for doc in raw_results['documents'][0]]

rerank_results = rerank_service.get_rerank(
    query_text=user_query,
    documents=rerank_inputs,
    top_n=config.retrieval.product.k_rerank 
)
print(f"=== [LẦN 2] KẾT QUẢ SAU KHI XẾP HẠNG LẠI BẰNG RERANK (K_RERANK = {config.retrieval.product.k_rerank}) ===")
for i, res in enumerate(rerank_results):
    
    orig_idx = res["index"]
    doc_id = raw_results['ids'][0][orig_idx]
    metadata = raw_results['metadatas'][0][orig_idx]
    score = res["relevance_score"]
    document_content = raw_results['documents'][0][orig_idx]
    
    # Trích xuất tên sản phẩm từ dòng đầu tiên của document_content
    product_name = document_content.split('\n')[0].replace("Sản phẩm:", "").strip()
    
    print(f"Top {i+1} [ID: {doc_id}] - Điểm liên quan (Rerank Score): {score:.4f}")
    print(f"=> Tên sản phẩm: {product_name}")  # Đã sửa
    print(f"=> Giá bán: {metadata.get('price')} VND")
    print(f"=> Nội dung chunk:\n{document_content}")
    print("-" * 50)

=== [LẦN 1] KẾT QUẢ TRUY VẤN THÔ TỪ CHROMADB (K_QUERY = 15) ===
Top 1 [ID: prod_127303] - Độ tương đồng (Cosine Distance): 1.0476
=> Tên sản phẩm: Laptop HP 14‑EM0023AU D0BG7PA
=> Giá bán: 19890000.0 VND
=> Nội dung chunk:
Sản phẩm: Laptop HP 14‑EM0023AU D0BG7PA

Thương hiệu: HP | Danh mục: laptop

Thông tin giá & Kho hàng:
- Giá thực tế: 19,890,000 VNĐ
- Giá gốc niêm yết: 23,990,000 VNĐ
- Mức giảm giá: 17.09%
- Tình trạng: Còn hàng (381 sản phẩm)

Thông số kỹ thuật chi tiết:
- Bộ vi xử lý (CPU/Chipset): AMD Ryzen 5 7520U (up to 4.3 GHz max boost clock, 4 MB L3 cache, 4 lõi, 8 luồng)
- Dung lượng RAM: 16GB
- Chuẩn/Loại RAM: DDR4-3200 MT/s Onboard
- Dung lượng lưu trữ: 512 GB PCIe NVMeTM M.2 SSD
- Kích thước màn hình: 14 inches
- Độ phân giải màn hình: 1920 x 1080 pixels (FullHD)
- Công nghệ màn hình: Màn hình chống chói
Độ sáng 250 nits
Độ phủ màu 45% NTSC
- Loại tấm nền màn hình: Màn hình chống chói
Độ sáng 250 nits
Độ phủ màu 45% NTSC
- Dung lượng Pin: 3-cell, 41 Wh Li-ion polymer
- 

In [7]:
class RAGService:
    def __init__(self, config, EmbeddingService, VectorDatabase, settings, api_service):
        self.config = config
        self.EmbeddingService = EmbeddingService
        self.VectorDatabase = VectorDatabase

        self.openrouter_key = settings.OPENROUTER_API_KEY
        self.gemini_key = settings.GEMINI_API_KEY
        self.groq_key = settings.GROQ_API_KEY

        self.model_name